# Bronze to silver cleaning

This notebook reads the bronze transit and weather tables, applies table-specific cleaning logic, and writes curated outputs to `dev.silver`.

In [0]:
from pyspark.sql import functions as F, Window

catalog_name = "dev"
bronze_schema = "bronze"
silver_schema = "silver"
checkpoint_base = f"/Volumes/{catalog_name}/{silver_schema}/checkpoints"

Databricks data profile. Run in Databricks to view.

In [0]:
def write_deduplicated_batch(batch_df, batch_id, key_columns, target_table_name):
    window_spec = Window.partitionBy(*key_columns).orderBy(F.col("ingested_at").desc())

    latest_batch = (
        batch_df
        .withColumn("row_num", F.row_number().over(window_spec))
        .filter(F.col("row_num") == 1)
        .drop("row_num")
    )

    (
        latest_batch.write
        .format("delta")
        .mode("append")
        .saveAsTable(f"{catalog_name}.{silver_schema}.{target_table_name}")
    )

In [0]:
def process_bronze_stream(
    source_table_name: str,
    target_table_name: str,
    key_columns,
    required_columns,
    extra_filter=None,
):
    stream_df = spark.readStream.table(f"{catalog_name}.{bronze_schema}.{source_table_name}")

    cleaned_df = stream_df
    for column in required_columns:
        cleaned_df = cleaned_df.filter(F.col(column).isNotNull())

    if extra_filter is not None:
        cleaned_df = cleaned_df.filter(extra_filter)

    query = (
        cleaned_df.writeStream
        .foreachBatch(
            lambda batch_df, batch_id: write_deduplicated_batch(
                batch_df,
                batch_id,
                key_columns,
                target_table_name,
            )
        )
        .option("checkpointLocation", f"{checkpoint_base}/{target_table_name}")
        .queryName(target_table_name)
        .start()
    )
    return query

## Vehicle positions

Common streaming pattern:
* read the bronze table as a stream
* filter invalid records
* deduplicate the latest batch using the latest `ingested_at`
* append the cleaned latest batch to the silver table

In [0]:
vehicle_positions_cleaned = process_bronze_stream(
    source_table_name="ttc_vehicle_positions_bronze",
    target_table_name="ttc_vehicle_positions_silver",
    key_columns=["event_id", "event_timestamp"],
    required_columns=["event_id", "vehicle_id", "trip_id", "event_timestamp"]
)

## Trip updates

Common streaming pattern:
* read the bronze table as a stream
* filter invalid records
* deduplicate the latest batch using the latest `ingested_at`
* append the cleaned latest batch to the silver table

In [0]:
trip_updates_cleaned = process_bronze_stream(
    source_table_name="ttc_trip_updates_bronze",
    target_table_name="ttc_trip_updates_silver",
    key_columns=["event_id", "event_timestamp"],
    required_columns=["event_id", "trip_id", "event_timestamp"],
    extra_filter=F.col("arrival_time").isNotNull() | F.col("departure_time").isNotNull(),
)

## Alerts

Common streaming pattern:
* read the bronze table as a stream
* filter invalid records
* deduplicate the latest batch using the latest `ingested_at`
* append the cleaned latest batch to the silver table

In [0]:
alerts_cleaned = process_bronze_stream(
    source_table_name="ttc_alerts_bronze",
    target_table_name="ttc_alerts_silver",
    key_columns=["event_id"],
    required_columns=["event_id", "alert_id", "message"]
)

Databricks data profile. Run in Databricks to view.

## Weather

Common streaming pattern:
* read the bronze table as a stream
* filter invalid records
* deduplicate the latest batch using the latest `ingested_at`
* append the cleaned latest batch to the silver table

In [0]:
weather_cleaned = process_bronze_stream(
    source_table_name="weather_bronze",
    target_table_name="weather_silver",
    key_columns=["city", "localtime"],
    required_columns=["city", "country", "localtime"],
)